## Obtenemos el histórico de constituyentes del S&P500 y lo guardamos en un CSV

In [ ]:
import pandas as pd
import requests

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}

# 1. Hacer la petición a Wikipedia
resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

# 2. Extraer las tablas HTML
tables = pd.read_html(resp.text)

# Tabla 1: Cambios históricos (es la segunda tabla en la página de Wikipedia)
df_chg = tables[1].copy()

# 3. Aplanar columnas si vienen como MultiIndex (como ocurre en Wikipedia)
if isinstance(df_chg.columns, pd.MultiIndex):
    df_chg.columns = [" ".join([str(x) for x in c if str(x) != "nan"]).strip().lower() for c in df_chg.columns]
else:
    df_chg.columns = [str(c).strip().lower() for c in df_chg.columns]

# 4. Detectar dinámicamente las columnas usando tu lógica original
date_col = next(c for c in df_chg.columns if "date" in c)
add_col = next(c for c in df_chg.columns if ("added" in c and "ticker" in c) or c == "added")
rem_col = next(c for c in df_chg.columns if ("removed" in c and "ticker" in c) or c == "removed")

# 5. Filtrar solo las columnas que nos interesan
df_final = df_chg[[date_col, add_col, rem_col]].copy()

# 6. Renombrar las columnas al formato exacto que solicitaste
df_final.columns = ['date', 'Tickr added', 'Tickr removed']

# 7. Limpiar la columna de fechas y ordenar
df_final['date'] = pd.to_datetime(df_final['date'], errors="coerce")
df_final = df_final.sort_values('date', ascending=False).reset_index(drop=True)

# 8. Guardar en un archivo CSV
nombre_archivo = "sp500_historico_cambios.csv"
df_final.to_csv(nombre_archivo, index=False)

C:\Users\jpuerta\AppData\Local\Temp\ipykernel_800\2304542349.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text)


¡Listo! Se han guardado 389 registros en el archivo 'sp500_historico_cambios.csv'.

Primeras 5 filas del documento guardado:
        date Tickr added Tickr removed
0 2026-02-09        CIEN           DAY
1 2025-12-22         FIX           MHK
2 2025-12-22         CRH           LKQ
3 2025-12-22        CVNA          SOLS
4 2025-12-11        ARES             K
